In [40]:
import os
import json
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as T
from tqdm import tqdm
import time
class ShadowDataset(Dataset):
    def __init__(self, folder, img_size=224):
        self.folder = folder
        self.img_size = img_size
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],  # ImageNet mean
                        [0.229, 0.224, 0.225])   # ImageNet std
        ])
        
        # Collect all images that have a matching json
        self.samples = [
            f.replace(".png", "")
            for f in os.listdir(folder)
            if f.endswith(".png") and 
               os.path.exists(os.path.join(folder, f.replace(".png", ".json")))
        ]
        print(f"Found {len(self.samples)} samples in {folder}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        
        # ── Load image ──
        img_path = os.path.join(self.folder, f"{name}.png")
        img = Image.open(img_path).convert("RGB")
        W, H = img.size          # original size, needed for normalizing
        img = self.transform(img)
        
        # ── Load annotation ──
        json_path = os.path.join(self.folder, f"{name}.json")
        with open(json_path) as f:
            ann = json.load(f)
        
        # Normalize all corner coordinates to 0-1
        # Note: values CAN be > 1.0 since person is off-screen!
        bbox = ann["bbox"]
        corners = torch.tensor([
            bbox["top_left"][0]     / W,
            bbox["top_left"][1]     / H,
            bbox["top_right"][0]    / W,
            bbox["top_right"][1]    / H,
            bbox["bottom_left"][0]  / W,
            bbox["bottom_left"][1]  / H,
            bbox["bottom_right"][0] / W,
            bbox["bottom_right"][1] / H,
        ], dtype=torch.float32)

        direction = torch.tensor(
            [ann["walking_into_frame_bool"]], 
        dtype=torch.float32
)
        
        return img, corners, direction, W, H, name


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# Load dataset
dataset = ShadowDataset("data/train_data/train_data")

# Split
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=32)

# Model
model = ShadowDetector().to(device)

# Loss + optimizer
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

best_val_loss = float("inf")

for epoch in range(50):
    start = time.time()

    # ── TRAIN ──
    model.train()
    train_losses = []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/50 [Train]", leave=False)
    for imgs, corners, _, _, _, _ in loop:
        imgs    = imgs.to(device)
        corners = corners.to(device)

        optimizer.zero_grad()

        pred_corners, _ = model(imgs)   # ignore direction output

        loss = criterion(pred_corners, corners)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    # ── VALIDATE ──
    model.eval()
    val_losses = []

    loop = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/50 [Val]", leave=False)
    with torch.no_grad():
        for imgs, corners, _, _, _, _ in loop:
            imgs    = imgs.to(device)
            corners = corners.to(device)

            pred_corners, _ = model(imgs)

            loss = criterion(pred_corners, corners)

            val_losses.append(loss.item())
            loop.set_postfix(loss=f"{loss.item():.4f}")

    avg_train = sum(train_losses) / len(train_losses)
    avg_val   = sum(val_losses) / len(val_losses)
    elapsed   = time.time() - start

    print(f"Epoch {epoch+1:02d}/50 | Train: {avg_train:.4f} | Val: {avg_val:.4f} | Time: {elapsed:.1f}s")

    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")
        print("↑ Saved best model")

    scheduler.step()

    model.load_state_dict(torch.load("best_model.pth"))
model.eval()

iou_scores = []

with torch.no_grad():
    for imgs, corners, _, W, H, _ in val_loader:
        imgs    = imgs.to(device)
        corners = corners.to(device)

        pred_corners, _ = model(imgs)

        # scale to pixels
        W = W.float().to(device).unsqueeze(1)
        H = H.float().to(device).unsqueeze(1)
        scale = torch.cat([W, H], dim=1).repeat(1, 4)

        pred_px = (pred_corners * scale).view(-1, 4, 2)
        true_px = (corners      * scale).view(-1, 4, 2)

        # bounding boxes
        pred_xmin = pred_px[:, :, 0].min(dim=1).values
        pred_ymin = pred_px[:, :, 1].min(dim=1).values
        pred_xmax = pred_px[:, :, 0].max(dim=1).values
        pred_ymax = pred_px[:, :, 1].max(dim=1).values

        true_xmin = true_px[:, :, 0].min(dim=1).values
        true_ymin = true_px[:, :, 1].min(dim=1).values
        true_xmax = true_px[:, :, 0].max(dim=1).values
        true_ymax = true_px[:, :, 1].max(dim=1).values

        # intersection
        inter_xmin = torch.max(pred_xmin, true_xmin)
        inter_ymin = torch.max(pred_ymin, true_ymin)
        inter_xmax = torch.min(pred_xmax, true_xmax)
        inter_ymax = torch.min(pred_ymax, true_ymax)

        inter_w = (inter_xmax - inter_xmin).clamp(min=0)
        inter_h = (inter_ymax - inter_ymin).clamp(min=0)
        intersection = inter_w * inter_h

        # union
        pred_area = (pred_xmax - pred_xmin) * (pred_ymax - pred_ymin)
        true_area = (true_xmax - true_xmin) * (true_ymax - true_ymin)
        union = pred_area + true_area - intersection

        iou = intersection / union.clamp(min=1e-6)

        iou_scores.extend(iou.cpu().tolist())

avg_iou = sum(iou_scores) / len(iou_scores)

print(f"Avg IoU: {avg_iou:.3f}")

Using: cuda
Found 1692 samples in data/train_data/train_data


KeyboardInterrupt: 

In [41]:
import os
import json
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as T
from tqdm import tqdm
import time

In [42]:
import torchvision.models as models

class ShadowDetector(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Pretrained ResNet18 backbone
        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        
        # Remove final classification layer
        self.backbone.fc = nn.Identity()
        
        # Regression head (8 values = 4 corners)
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 8)  # 4 corners (x,y)
        )

    def forward(self, x):
        features = self.backbone(x)
        corners = self.head(features)
        return corners, None  # keep your interface consistent

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# Load dataset
dataset = ShadowDataset("data/train_data/train_data")

# Split
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=32)

# Model
model = ShadowDetector().to(device)

# Loss + optimizer
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

best_val_loss = float("inf")

for epoch in range(50):
    start = time.time()

    # ── TRAIN ──
    model.train()
    train_losses = []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/50 [Train]", leave=False)
    for imgs, corners, _, _, _, _ in loop:
        imgs    = imgs.to(device)
        corners = corners.to(device)

        optimizer.zero_grad()

        pred_corners, _ = model(imgs)   # ignore direction output

        loss = criterion(pred_corners, corners)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    # ── VALIDATE ──
    model.eval()
    val_losses = []

    loop = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/50 [Val]", leave=False)
    with torch.no_grad():
        for imgs, corners, _, _, _, _ in loop:
            imgs    = imgs.to(device)
            corners = corners.to(device)

            pred_corners, _ = model(imgs)

            loss = criterion(pred_corners, corners)

            val_losses.append(loss.item())
            loop.set_postfix(loss=f"{loss.item():.4f}")

    avg_train = sum(train_losses) / len(train_losses)
    avg_val   = sum(val_losses) / len(val_losses)
    elapsed   = time.time() - start

    print(f"Epoch {epoch+1:02d}/50 | Train: {avg_train:.4f} | Val: {avg_val:.4f} | Time: {elapsed:.1f}s")

    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")
        print("↑ Saved best model")

    scheduler.step()

Using: cuda
Found 1692 samples in data/train_data/train_data


Epoch 01/50 | Train: 0.0342 | Val: 0.0045 | Time: 19.3s
↑ Saved best model


Epoch 02/50 | Train: 0.0093 | Val: 0.0042 | Time: 23.1s
↑ Saved best model


Epoch 03/50 | Train: 0.0078 | Val: 0.0014 | Time: 27.1s
↑ Saved best model


Epoch 04/50 | Train: 0.0057 | Val: 0.0071 | Time: 27.0s


...

Epoch 19/50 | Train: 0.0029 | Val: 0.0025 | Time: 19.0s


Epoch 20/50 | Train: 0.0027 | Val: 0.0009 | Time: 19.4s


Epoch 21/50 | Train: 0.0028 | Val: 0.0006 | Time: 19.0s


Epoch 22/50 | Train: 0.0028 | Val: 0.0006 | Time: 19.8s


Epoch 23/50 [Val]:  18%|█▊        | 2/11 [00:00<00:02,  3.24it/s, loss=0.0007]   

In [20]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

iou_scores = []

with torch.no_grad():
    for imgs, corners, _, W, H, _ in val_loader:
        imgs    = imgs.to(device)
        corners = corners.to(device)

        pred_corners, _ = model(imgs)

        # scale to pixels
        W = W.float().to(device).unsqueeze(1)
        H = H.float().to(device).unsqueeze(1)
        scale = torch.cat([W, H], dim=1).repeat(1, 4)

        pred_px = (pred_corners * scale).view(-1, 4, 2)
        true_px = (corners      * scale).view(-1, 4, 2)

        # bounding boxes
        pred_xmin = pred_px[:, :, 0].min(dim=1).values
        pred_ymin = pred_px[:, :, 1].min(dim=1).values
        pred_xmax = pred_px[:, :, 0].max(dim=1).values
        pred_ymax = pred_px[:, :, 1].max(dim=1).values

        true_xmin = true_px[:, :, 0].min(dim=1).values
        true_ymin = true_px[:, :, 1].min(dim=1).values
        true_xmax = true_px[:, :, 0].max(dim=1).values
        true_ymax = true_px[:, :, 1].max(dim=1).values

        # intersection
        inter_xmin = torch.max(pred_xmin, true_xmin)
        inter_ymin = torch.max(pred_ymin, true_ymin)
        inter_xmax = torch.min(pred_xmax, true_xmax)
        inter_ymax = torch.min(pred_ymax, true_ymax)

        inter_w = (inter_xmax - inter_xmin).clamp(min=0)
        inter_h = (inter_ymax - inter_ymin).clamp(min=0)
        intersection = inter_w * inter_h

        # union
        pred_area = (pred_xmax - pred_xmin) * (pred_ymax - pred_ymin)
        true_area = (true_xmax - true_xmin) * (true_ymax - true_ymin)
        union = pred_area + true_area - intersection

        iou = intersection / union.clamp(min=1e-6)

        iou_scores.extend(iou.cpu().tolist())

avg_iou = sum(iou_scores) / len(iou_scores)

print(f"Avg IoU: {avg_iou:.3f}")

Avg IoU: 0.575


In [22]:
import os
import json
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as T
import torchvision.models as models

from tqdm import tqdm
import time

In [23]:
class ShadowDataset(Dataset):
    def __init__(self, folder, img_size=224):
        self.folder = folder
        self.img_size = img_size

        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],
                        [0.229, 0.224, 0.225])
        ])

        self.samples = [
            f.replace(".png", "")
            for f in os.listdir(folder)
            if f.endswith(".png") and 
               os.path.exists(os.path.join(folder, f.replace(".png", ".json")))
        ]

        print(f"Found {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]

        img_path = os.path.join(self.folder, f"{name}.png")
        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        img = self.transform(img)

        json_path = os.path.join(self.folder, f"{name}.json")
        with open(json_path) as f:
            ann = json.load(f)

        bbox = ann["bbox"]

        x_coords = [bbox["top_left"][0], bbox["top_right"][0],
                    bbox["bottom_left"][0], bbox["bottom_right"][0]]

        y_coords = [bbox["top_left"][1], bbox["top_right"][1],
                    bbox["bottom_left"][1], bbox["bottom_right"][1]]

        xmin, xmax = min(x_coords), max(x_coords)
        ymin, ymax = min(y_coords), max(y_coords)

        cx = ((xmin + xmax) / 2) / W
        cy = ((ymin + ymax) / 2) / H
        w  = (xmax - xmin) / W
        h  = (ymax - ymin) / H

        box = torch.tensor([cx, cy, w, h], dtype=torch.float32)

        return img, box, W, H

In [24]:
class ShadowDetector(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone.fc = nn.Identity()

        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 4)
        )

    def forward(self, x):
        features = self.backbone(x)
        box = self.head(features)

        # stabilize width/height
        box[:, 2:] = torch.relu(box[:, 2:])

        return box

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

dataset = ShadowDataset("data/train_data/train_data")

train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size

train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=32)

model = ShadowDetector().to(device)

criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

best_val_loss = float("inf")

Using: cuda
Found 1692 samples


In [30]:
for epoch in range(50):
    start = time.time()

    # TRAIN
    model.train()
    train_losses = []

    for imgs, boxes, _, _ in tqdm(train_loader, desc=f"Epoch {epoch+1:02d} Train"):
        imgs  = imgs.to(device)
        boxes = boxes.to(device)

        optimizer.zero_grad()

        pred_boxes = model(imgs)

        loss = criterion(pred_boxes, boxes)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # VALIDATE
    model.eval()
    val_losses = []

    with torch.no_grad():
        for imgs, boxes, _, _ in tqdm(val_loader, desc=f"Epoch {epoch+1:02d} Val"):
            imgs  = imgs.to(device)
            boxes = boxes.to(device)

            pred_boxes = model(imgs)

            loss = criterion(pred_boxes, boxes)
            val_losses.append(loss.item())

    avg_train = sum(train_losses) / len(train_losses)
    avg_val   = sum(val_losses) / len(val_losses)

    print(f"Epoch {epoch+1:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")
        print("↑ Saved best model")

    scheduler.step()

Epoch 01 Val: 100%|██████████| 11/11 [00:04<00:00,  2.40it/s]


Epoch 01 | Train: 0.0077 | Val: 0.0919


Epoch 02 Train:   9%|▉         | 4/43 [00:02<00:22,  1.76it/s]


KeyboardInterrupt: 

In [29]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

iou_scores = []

with torch.no_grad():
    for imgs, boxes, W, H in val_loader:
        imgs  = imgs.to(device)
        boxes = boxes.to(device)

        pred_boxes = model(imgs)

        # Convert to corners
        cx, cy, w, h = pred_boxes[:,0], pred_boxes[:,1], pred_boxes[:,2], pred_boxes[:,3]
        px_min = cx - w/2
        px_max = cx + w/2
        py_min = cy - h/2
        py_max = cy + h/2

        tx, ty, tw, th = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
        tx_min = tx - tw/2
        tx_max = tx + tw/2
        ty_min = ty - th/2
        ty_max = ty + th/2

        inter_xmin = torch.max(px_min, tx_min)
        inter_ymin = torch.max(py_min, ty_min)
        inter_xmax = torch.min(px_max, tx_max)
        inter_ymax = torch.min(py_max, ty_max)

        inter_w = (inter_xmax - inter_xmin).clamp(min=0)
        inter_h = (inter_ymax - inter_ymin).clamp(min=0)
        intersection = inter_w * inter_h

        pred_area = (px_max - px_min) * (py_max - py_min)
        true_area = (tx_max - tx_min) * (ty_max - ty_min)

        union = pred_area + true_area - intersection

        iou = intersection / union.clamp(min=1e-6)

        iou_scores.extend(iou.cpu().tolist())

avg_iou = sum(iou_scores) / len(iou_scores)

print(f"🔥 Avg IoU: {avg_iou:.3f}")

🔥 Avg IoU: 0.228


In [28]:
import torch
import time
from tqdm import tqdm

best_val_loss = float("inf")

for epoch in range(50):
    start = time.time()

    # =========================
    # TRAIN
    # =========================
    model.train()
    train_losses = []

    for imgs, boxes, _, _ in tqdm(train_loader, desc=f"Epoch {epoch+1:02d} Train"):
        imgs = imgs.to(device)
        boxes = boxes.to(device)

        optimizer.zero_grad()

        pred_boxes = model(imgs)

        # 🔥 CLAMP predictions to keep valid geometry
        pred_boxes = torch.clamp(pred_boxes, 0.0, 1.0)

        loss = criterion(pred_boxes, boxes)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # stability fix
        optimizer.step()

        train_losses.append(loss.item())

    # =========================
    # VALIDATION
    # =========================
    model.eval()
    val_losses = []

    with torch.no_grad():
        for imgs, boxes, _, _ in tqdm(val_loader, desc=f"Epoch {epoch+1:02d} Val"):
            imgs = imgs.to(device)
            boxes = boxes.to(device)

            pred_boxes = model(imgs)
            pred_boxes = torch.clamp(pred_boxes, 0.0, 1.0)

            loss = criterion(pred_boxes, boxes)
            val_losses.append(loss.item())

    avg_train = sum(train_losses) / len(train_losses)
    avg_val = sum(val_losses) / len(val_losses)

    print(f"Epoch {epoch+1:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")

    # save best
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pth")
        print("↑ Saved best model")

    # scheduler fix (only if ReduceLROnPlateau)
    scheduler.step(avg_val)

    print(f"Time: {time.time() - start:.2f}s")

Epoch 01 Val: 100%|██████████| 11/11 [00:03<00:00,  3.05it/s]
c:\Users\prosh\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:243: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch 01 | Train: 0.0088 | Val: 0.0076
↑ Saved best model
Time: 21.00s


Epoch 02 Val: 100%|██████████| 11/11 [00:03<00:00,  3.03it/s]


Epoch 02 | Train: 0.0081 | Val: 0.0079
Time: 20.18s


Epoch 03 Val: 100%|██████████| 11/11 [00:03<00:00,  2.94it/s]


...

Epoch 09 | Train: 0.0088 | Val: 0.0089
Time: 22.63s


Epoch 10 Val: 100%|██████████| 11/11 [00:04<00:00,  2.64it/s]


Epoch 10 | Train: 0.0108 | Val: 0.0550
Time: 22.21s


Epoch 11 Val:  82%|████████▏ | 9/11 [00:03<00:00,  2.33it/s]


KeyboardInterrupt: 